In [44]:
import sys
!{sys.executable} -m pip install numpy torch torchvision mediapipe opencv-python scikit-learn


[notice] A new release of pip is available: 23.1.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [45]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import confusion_matrix
import json, os, random

In [46]:
TARGET_FRAMES = 15
CLASSES       = ["swipe_right", "swipe_left", "null", "swipe_up", "swipe_down", "tap"]
BATCH_SIZE    = 32
EPOCHS        = 60
LR            = 1e-3
HIDDEN        = 128
NUM_LAYERS    = 2
VAL_SPLIT     = 0.20
DATA_PATH     = "data"

In [47]:
def interpolate_frames(frames, target=TARGET_FRAMES):
    """Resize any sequence length to exactly TARGET_FRAMES via linear interpolation.
    Better than truncating (which cuts off gesture endings) or padding (which
    adds dead frames). Works for both longer and shorter sequences."""
    frames = np.array(frames, dtype=np.float32)   # (T, 21, 3)
    T = len(frames)
    if T == target:
        return frames
    old_idx = np.linspace(0, T - 1, T)
    new_idx = np.linspace(0, T - 1, target)
    out = np.zeros((target, 21, 3), dtype=np.float32)
    for j in range(21):
        for k in range(3):
            out[:, j, k] = np.interp(new_idx, old_idx, frames[:, j, k])
    return out

In [48]:
def normalize_landmarks(frames):
    """Wrist-relative + scale normalisation — must match inference exactly."""
    frames = frames.copy()
    frames[:, :, :2] -= frames[:, 0:1, :2]          # wrist at origin
    scale = np.linalg.norm(
        frames[:, 9, :2] - frames[:, 0, :2], axis=-1, keepdims=True
    )
    scale = np.maximum(scale, 1e-6)
    frames[:, :, :2] /= scale[:, np.newaxis]
    return frames

In [49]:
def load_sample(path):
    frames = json.load(open(path))["frames"]
    frames = interpolate_frames(frames)              # (30, 21, 3)
    frames = normalize_landmarks(frames)
    return frames.reshape(TARGET_FRAMES, -1).astype(np.float32)  # (30, 63)

In [50]:
class GestureDataset(Dataset):
    def __init__(self, samples):
        """samples: list of (path, label_index)"""
        self.samples    = samples
 
    def __len__(self):
        return len(self.samples)
 
    def __getitem__(self, i):
        path, label = self.samples[i]
        x = load_sample(path)
        return torch.tensor(x), torch.tensor(label)

In [51]:
def build_splits(data_path, val_split=VAL_SPLIT):
    """Load all samples, do a stratified train/val split."""
    all_samples = []
    for label_idx, gesture in enumerate(CLASSES):
        folder = f"{data_path}/{gesture}"
        if not os.path.exists(folder):
            print(f"Warning: {folder} not found, skipping")
            continue
        files = [f for f in os.listdir(folder) if f.endswith(".json")]
        for fname in files:
            all_samples.append((f"{folder}/{fname}", label_idx))
 
    # Stratified split — preserve class ratios in both sets
    from collections import defaultdict
    by_class = defaultdict(list)
    for s in all_samples:
        by_class[s[1]].append(s)
 
    train_samples, val_samples = [], []
    for label_idx, samples in by_class.items():
        random.shuffle(samples)
        n_val = max(1, int(len(samples) * val_split))
        val_samples  += samples[:n_val]
        train_samples += samples[n_val:]
 
    print(f"Train: {len(train_samples)} | Val: {len(val_samples)}")
    for c_idx, c in enumerate(CLASSES):
        tr = sum(1 for _, l in train_samples if l == c_idx)
        va = sum(1 for _, l in val_samples   if l == c_idx)
        print(f"  {c:>12}  train={tr}  val={va}")
 
    return train_samples, val_samples

In [52]:
class GestureLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(63, HIDDEN, NUM_LAYERS,
                            batch_first=True, dropout=0.3)
        self.head = nn.Sequential(
            nn.Linear(HIDDEN, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, len(CLASSES))
        )
 
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.head(h[-1])

In [53]:
def make_sampler(samples):
    labels = [l for _, l in samples]
    counts = np.bincount(labels, minlength=len(CLASSES)).astype(float)
    weights_per_class = 1.0 / np.maximum(counts, 1)
    sample_weights = [weights_per_class[l] for l in labels]
    return WeightedRandomSampler(sample_weights, len(sample_weights))

In [54]:
train_samples, val_samples = build_splits(DATA_PATH)

train_ds = GestureDataset(train_samples)
val_ds   = GestureDataset(val_samples)

Train: 266 | Val: 63
   swipe_right  train=27  val=6
    swipe_left  train=27  val=6
          null  train=128  val=32
      swipe_up  train=27  val=6
    swipe_down  train=27  val=6
           tap  train=30  val=7


In [55]:
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE,
                        sampler=make_sampler(train_samples))
val_dl   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In [56]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")


Device: cpu


In [57]:
model     = GestureLSTM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=7, factor=0.5, verbose=True
)

best_val_loss = float("inf")
patience_counter = 0
EARLY_STOP_PATIENCE = 15

print(f"\nTraining for up to {EPOCHS} epochs (early stop patience={EARLY_STOP_PATIENCE})\n")


Training for up to 60 epochs (early stop patience=15)



/Users/yoraiyaniv/CODE/GesturesDetection/.env/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


In [58]:
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_dl:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    model.eval()
    val_loss, correct = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_dl:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            out      = model(X_batch)
            val_loss += criterion(out, y_batch).item()
            correct  += (out.argmax(1) == y_batch).sum().item()

    val_acc   = correct / len(val_ds)
    val_loss /= len(val_dl)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:3d} | "
            f"train {train_loss/len(train_dl):.3f} | "
            f"val {val_loss:.3f} | "
            f"acc {val_acc:.2%}")
    
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "gesture_model.pt")
        print(f"  ✓ saved  (val_loss={best_val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

Epoch   1 | train 1.777 | val 1.803 | acc 17.46%
  ✓ saved  (val_loss=1.8027)
Epoch   2 | train 1.672 | val 1.670 | acc 22.22%
  ✓ saved  (val_loss=1.6703)
Epoch   3 | train 1.369 | val 1.431 | acc 44.44%
  ✓ saved  (val_loss=1.4313)
Epoch   4 | train 1.073 | val 1.318 | acc 46.03%
  ✓ saved  (val_loss=1.3177)
Epoch   5 | train 0.868 | val 1.152 | acc 49.21%
  ✓ saved  (val_loss=1.1520)
Epoch   6 | train 0.641 | val 1.077 | acc 49.21%
  ✓ saved  (val_loss=1.0770)
Epoch   7 | train 0.565 | val 0.949 | acc 52.38%
  ✓ saved  (val_loss=0.9490)
Epoch   8 | train 0.532 | val 1.130 | acc 49.21%
Epoch   9 | train 0.637 | val 1.074 | acc 50.79%
Epoch  10 | train 0.548 | val 1.088 | acc 49.21%
Epoch  11 | train 0.476 | val 1.022 | acc 50.79%
Epoch  12 | train 0.446 | val 0.962 | acc 52.38%
Epoch  13 | train 0.400 | val 1.086 | acc 55.56%
Epoch  14 | train 0.396 | val 1.061 | acc 57.14%
Epoch  15 | train 0.398 | val 1.155 | acc 57.14%
Epoch  16 | train 0.386 | val 0.919 | acc 57.14%
  ✓ saved  (v

In [59]:
print("\n── Final evaluation on val set ──────────────────────────────────")
model.load_state_dict(torch.load("gesture_model.pt", map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in val_dl:
        preds = model(X_batch.to(device)).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(y_batch.tolist())

cm = confusion_matrix(all_labels, all_preds)
print(f"\n{'':>12}", "  ".join(f"{c:>12}" for c in CLASSES))
for i, row in enumerate(cm):
    print(f"{CLASSES[i]:>12}", "  ".join(f"{v:>12}" for v in row))

overall_acc = sum(a == p for a, p in zip(all_labels, all_preds)) / len(all_labels)
print(f"\nOverall val accuracy: {overall_acc:.2%}")


── Final evaluation on val set ──────────────────────────────────

              swipe_right    swipe_left          null      swipe_up    swipe_down           tap
 swipe_right            5             0             1             0             0             0
  swipe_left            0             5             1             0             0             0
        null            0             0            19             6             4             3
    swipe_up            0             0             0             6             0             0
  swipe_down            0             0             0             0             6             0
         tap            0             0             0             0             0             7

Overall val accuracy: 76.19%


In [60]:
def predict(json_path, model, device):
    sample = load_sample(json_path)                       # (30, 63)
    x = torch.tensor(sample).unsqueeze(0).to(device)     # (1, 30, 63)
 
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
        pred  = probs.argmax().item()
 
    print(f"Prediction : {CLASSES[pred]}")
    print(f"Confidence : {probs[pred]:.2%}")
    print(f"All probs  : { {c: f'{p:.2%}' for c, p in zip(CLASSES, probs.tolist())} }")

In [63]:
predict("data/swipe_left//sample_010.json", model, device)

Prediction : swipe_left
Confidence : 62.90%
All probs  : {'swipe_right': '0.18%', 'swipe_left': '62.90%', 'null': '36.74%', 'swipe_up': '0.01%', 'swipe_down': '0.02%', 'tap': '0.15%'}


In [62]:
torch.save(model.state_dict(), "gesture_model.pt")